# AKORI · Notebook DEMO

À lancer pour la démonstration, **après** le notebook Build. Rien n'est recalculé : les index, résumés, flashcards et quiz
sont déjà sur Google Drive ; les index sont chargés en mémoire au démarrage.

## Comment accéder à la console d'administration

1. Lancez la cellule 4. Au tout premier lancement, AKORI crée le compte **administrateur** et affiche son mot de passe dans
   la sortie de la cellule (il est aussi enregistré dans `MyDrive/AKORI/ADMIN_INITIAL_PASSWORD.txt`).
   Pour choisir vous-même ce mot de passe, ajoutez le secret Colab `AKORI_ADMIN_PASSWORD` avant de lancer.
2. Ouvrez le lien `https://….gradio.live` affiché.
3. Sur l'écran de connexion, saisissez l'identifiant **`admin`** dans le champ *Email* et le mot de passe de l'administrateur.
4. Dans le menu de gauche, cliquez sur **Administration** (🛡). Ce bouton n'existe que pour un compte administrateur.
5. Pour la démonstration, connectez-vous plutôt avec le compte du notebook Build (`demo@akori.app`).

**Mot de passe administrateur oublié ?** Ajoutez les secrets `AKORI_ADMIN_PASSWORD` (nouveau mot de passe, 8 caractères minimum)
et `AKORI_RESET_ADMIN` = `1`, puis relancez la cellule 4.

## 1. Installation

In [ ]:
!pip -q install gradio google-genai pymupdf langchain-community langchain-text-splitters faiss-cpu sentence-transformers

## 2. Google Drive et clés

In [ ]:
import os
from google.colab import drive, userdata
drive.mount("/content/drive")                         # AKORI enregistre tout dans MyDrive/AKORI

def secret(name, default=""):
    try:
        return userdata.get(name) or default
    except Exception:
        return default

os.environ["GEMINI_API_KEY"] = secret("GEMINI_API_KEY")        # Colab > icône clé > Secrets
assert os.environ["GEMINI_API_KEY"], "Ajoutez le secret GEMINI_API_KEY (icône clé à gauche) puis relancez la cellule."

## 3. Récupérer le code d'AKORI

In [ ]:
import os, subprocess, shutil
os.chdir("/content")                                   # sortir du dossier avant de le supprimer
REPO   = "github.com/rafalimanananomeniavoedmjohana-bit/AKORI.git"
BRANCH = "claude/epic-fermat-mphgx3"                   # passez à "main" une fois la branche fusionnée
CODE_DIR = "/content/akori_code"
DRIVE_COPY = "/content/drive/MyDrive/AKORI/code/akori.py"   # solution de secours : akori.py copié sur Drive

# Dépôt privé : ajoutez un secret Colab GITHUB_TOKEN (GitHub > Settings > Developer settings > Fine-grained token, lecture seule du dépôt)
try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN") or ""
except Exception:
    token = ""

shutil.rmtree(CODE_DIR, ignore_errors=True)
url = f"https://{token + '@' if token else ''}{REPO}"
done = subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, url, CODE_DIR]).returncode == 0
if not done and os.path.exists(DRIVE_COPY):
    os.makedirs(CODE_DIR, exist_ok=True)
    shutil.copy(DRIVE_COPY, CODE_DIR)
    done = True
    print("Code copié depuis Google Drive.")
if not done:
    from google.colab import files
    os.makedirs(CODE_DIR, exist_ok=True)
    print("Dépôt inaccessible (privé ?). Importez le fichier akori.py (et lui seul).")
    for name, data in files.upload().items():
        if name.endswith(".py"):
            open(os.path.join(CODE_DIR, "akori.py"), "wb").write(data)
assert os.path.exists(os.path.join(CODE_DIR, "akori.py")), "akori.py introuvable : relancez la cellule et importez akori.py"
print("Code prêt :", os.listdir(CODE_DIR))


## 4. Lancer AKORI

In [ ]:
import os
# Réglages optionnels (secrets Colab, ou valeurs ci-dessous)
for name in ("AKORI_ADMIN_PASSWORD", "AKORI_RESET_ADMIN", "AKORI_MIN_SIMILARITY", "AKORI_EMBED_MODEL"):
    value = secret(name)
    if value:
        os.environ[name] = value

os.chdir(CODE_DIR)
!python akori.py